In [1]:
from aig_grapher import AIG
from typing import Dict, Set, List, Tuple

In [2]:
from pydantic import BaseModel

from pydantic_ai import Agent
from pydantic_ai import RunContext, UsageLimits
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider


local_model = OpenAIChatModel(
	model_name='qwen3-vl:8b',
	provider=OllamaProvider(base_url='http://kfed:11434/v1'),  
)

cloud_model = OpenAIChatModel(
	model_name="qwen3.5:cloud",
	provider=OllamaProvider(base_url='http://localhost:11434/v1'),  
)

local_model = cloud_model

In [13]:

class Config:
	def __init__(self, aig_path: str):
		self.aig_path: str = aig_path
		self.aig: AIG = AIG(aig_path)
		self.expert_comm_hist: List[str] = []
		self.score: int = 0
		self.opponent_stats: Tuple[int, str] = (0, "") # score, strategies
		self.strategies: List[str] = []
		self.knowledge_base: List[str] = []
		self.results: Set[Tuple[int, int]] = set()
		self.tried_tuples: Set[Tuple[int, int]] = set() 

	def dump(self, filename: str):
		"""Persist the current game state to disk.

		The dump excludes the in-memory AIG object (which is not JSON-serializable)
		and instead keeps the path to the original AIG file.
		"""
		import json

		data = {
			"aig_path": self.aig_path,
			"score": self.score,
			"opponent_stats": self.opponent_stats,
			"strategies": self.strategies,
			"results": [list(t) for t in self.results],
			"tried_tuples": [list(t) for t in self.tried_tuples],
			"expert_comm_hist": self.expert_comm_hist,
		}

		with open(filename, "w", encoding="utf-8") as f:
			json.dump(data, f, indent=2)

	@classmethod
	def load(cls, filename: str) -> "Config":
		"""Load a previously dumped game state from disk."""
		import json

		with open(filename, "r", encoding="utf-8") as f:
			data = json.load(f)

		conf = cls(data["aig_path"])
		conf.score = data.get("score", 0)
		conf.opponent_stats = tuple(data.get("opponent_stats", (0, "")))
		conf.strategies = data.get("strategies", [])
		conf.results = {tuple(t) for t in data.get("results", [])}
		conf.tried_tuples = {tuple(t) for t in data.get("tried_tuples", [])}
		conf.expert_comm_hist = data.get("expert_comm_hist", [])
		return conf

In [31]:
aig_agent_instr = """
You are a one of the two participants of an And-Inverter Graph Exploration game. You are given an AIG graph in ASCII format, and you have to find meaningful equivalences between nodes of the graph. 
For every invariant or equivalence you find, you will submit it as a response to a validator, who will award you points based on the following criteria:
- If the invariant/equivalence is correct and non-trivial, you will receive 10 points.
- If the invariant/equivalence is correct but trivial, you will receive 5 points
- If the invariant/equivalence is incorrect or if proving the correctness timeouts the validator, you will receive 0 points.
You are expected to find as many correct and non-trivial invariants/equivalences as possible to maximize your score. You can also find trivial equivalences, but they will yield fewer points.
IMPORTANT: 
	Since you *are* interrupted and restarted from blank state on some different graph at many points due to usage-based timeouts, it is advised to store your developed strategies and insights in persistent storage, and load them back when you are restarted, so that you can continue improving your strategies over time. 
    Always check for strategies and knowledge already stored.
SIDENOTE: You are allowed to cheat by reading from your opponent's progress and strategies, but the opponent isn't allowed to know that you are cheating. So, be discreet if you choose to cheat.
CONSTRAINT: You will only be allowed to generate 5000 tokens, so do efficient work and save all learnings for reuse in later turns.
"""

expert_agent_instr = """
You are an expert whose task is to aid a novice LLM whenever it needs help. As all communications with you will be a LLM<->LLM communication, you are free to use more efficient ways of information communication.
"""

In [45]:
aig_agent = Agent(
	model=local_model,
	system_prompt=aig_agent_instr,
	deps_type=Config
)

expert_agent = Agent(
	model=cloud_model,
	instructions=expert_agent_instr
)

# @aig_agent.tool_plain
# def get_weather(city: str) -> int:
#     """
#     Get weather for a given city, but with a twist. The returned value is 8 * current temp (in C)
#     Args:
#         city: name of the city
#     Returns:
#         temperature
#     """
#     return 100


@aig_agent.tool
def check_equivalence_and_add(ctx: RunContext[Config], a: int, b: int) -> Tuple[int, Tuple[int, int], str]:
    """
    Checks if nodes a and b are indeed equivalent and returns a score based on the correctness and triviality of the finding.
    Also, if indeed equivalent, this function stores the equivalence and updates the score.
    Every failed use, incurs a -3 penalty.
    Args:
        a, b: nodes in AIG graph (follows same conventions as node numbers in .aag/.aig files). 
    Returns:
        score: the reward/penalty for the equivalence check submission.
        simulation_tuple: (sim_val_a, sim_val_b) where each value is a N-bit integer corresponding to the value calculated for nodes a and b during a parallel N-bit simulation on the AIG. Returns negative values if function run on illegal inputs.
        err_msg: error message if any
    """
    conf = ctx.deps
    eqv_tuple = (a, b) if a > b else (b, a)
    err_msg_success = "please store your strategy asap"
    try:
        assert a != b, "same input node ids"
        assert eqv_tuple not in conf.tried_tuples, "simulation already tried for pair"

        conf.tried_tuples.add(eqv_tuple)
        aig = conf.aig 
        nv, _ = aig.simulate(32)
        sim_val_a = nv[a//2]
        sim_val_b = nv[b//2]
        if a % 2 == 1:
            sim_val_a = sim_val_a ^ ((1 << 32) - 1)
        if b % 2 == 1:
            sim_val_b = sim_val_b ^ ((1 << 32) - 1)
        score_change = 10 if sim_val_a == sim_val_b else 0
        conf.score += score_change
        if score_change != 0:
            conf.results.add(eqv_tuple)
        return score_change, (sim_val_a, sim_val_b), err_msg_success if score_change != 0 else "failed"
    except Exception as e:
        conf.score -= 3
        return 0, (-1, -1), str(type(e)) + " " + str(e) + " | If you find any reason for your failure, add that to knowledge_base"
  
# @aig_agent.tool_plain
# def expert_comm_history() -> List[str]:
#     """
#     Get your history of communication with the expert.
#     """
#     return C.expert_comm_hist

# @aig_agent.tool_plain
# async def expert_help(prompt: str) -> str:
#     """
#     Allows you to send a prompt to an expert for support when you are stuck or confused.
#     Args:
#         prompt: your question/doubts/queries. Make sure to tell the expert that you need the data for use by another LLM and not a human so that the communication is more efficient.
#     """
#     result = await expert_agent.run(prompt)
#     C.expert_comm_hist.append(result.output)
#     return result.output

@aig_agent.tool
def score(ctx: RunContext[Config]) -> int:
    """
    Returns your current game score that you have to maximize
    """
    return ctx.deps.score

@aig_agent.tool
def results(ctx: RunContext[Config]):
    """
    Returns your currently found equivalences/invariants.
    """
    return ctx.deps.results

# @aig_agent.tool_plain
# def opponent_stats() -> Tuple[int, str]:
#     """
#     Returns the current score and discovered strategies of the opponent
#     """
#     return C.opponent_stats

@aig_agent.tool
def append_strategies(ctx: RunContext[Config], strategy: List[str]):
    """
    Store new strategies in your private knowledge that helped you to find equivalences, or things that you learnt along the way by failures.
    """
    ctx.deps.strategies.extend(strategy)

@aig_agent.tool
def strategies(ctx: RunContext[Config]) -> List[str]:
    """
    Returns the stored strategies from this and past runs
    """
    return ctx.deps.strategies

@aig_agent.tool
def knowledge_base(ctx: RunContext[Config]) -> List[str]:
    """
    Returns your stored knowledge base from this and past runs
    """
    return ctx.deps.knowledge_base

@aig_agent.tool
def append_knowledge_base(ctx: RunContext[Config], knowledge: List[str]):
    """
    Store new generalized knowledge in your private knowledge base that applies on all AIG graphs
    Usage direction: Whenever you find any information, trivial or not, but important enough to get up to work faster next time you are started, store it here.
    """
    ctx.deps.knowledge_base.extend(knowledge)

In [46]:
def read_file_as_string(filepath: str) -> str:
	with open(filepath, "r", encoding="utf-8") as f:
		return f.read()

In [47]:
# from pydantic_ai import Agent

# async def debug_run():
#     # Use run_stream_events instead of run
#     async with aig_agent.run_stream_events("Your prompt here") as stream:
#         async for event in stream:
#             # You can see exactly what kind of event is happening
#             if event.event_kind == 'model_request':
#                 print("--- 🤖 Model is thinking... ---")
#             elif event.event_kind == 'call_tools':
#                 print(f"--- 🛠️ Calling Tools: {event.tool_calls} ---")
#             elif event.event_kind == 'tool_result':
#                 print(f"--- ✅ Tool Returned: {event.result} ---")

In [49]:
aig_path = "/home/krishnendu/Research/fv-invariant-mining/data/circuits/aag/miter_mult_3bit.aag"
aig_str = read_file_as_string(aig_path)
events = []
conf = Config(aig_path)
conf.strategies = old_conf.strategies
# conf.strategies.extend(old_conf.strategies)
limits = UsageLimits(
	# request_limit=10,          # Max 3 total turns (Thinking + Tool Call + Response)
	output_tokens_limit=5000,   # Stop if the total output across all turns exceeds 500
)
async for event in aig_agent.run_stream_events(aig_str, deps=conf, usage_limits=limits):
	events.append(event)
	if event.event_kind != 'part_delta':
		print(event)


PartStartEvent(index=0, part=ThinkingPart(content='I', id='reasoning', provider_name='ollama'))
PartEndEvent(index=0, part=ThinkingPart(content='I need to analyze this AIG (And-Inverter Graph) and find meaningful equivalences between nodes. Let me start by understanding the structure:\n\n- 129 gates, 6 inputs, 0 latches, 1 output, 123 AND gates\n- Inputs are: 2, 4, 6, 8, 10, 12\n- Output is: 259\n\nLet me first check what strategies and knowledge I already have stored from previous runs.\n', id='reasoning', provider_name='ollama'), next_part_kind='tool-call')
PartStartEvent(index=1, part=ToolCallPart(tool_name='strategies', args='{}', tool_call_id='call_bb83mcbc'), previous_part_kind='thinking')
PartEndEvent(index=1, part=ToolCallPart(tool_name='strategies', args='{}', tool_call_id='call_bb83mcbc'), next_part_kind='tool-call')
PartStartEvent(index=2, part=ToolCallPart(tool_name='knowledge_base', args='{}', tool_call_id='call_hsqgifrs'), previous_part_kind='tool-call')
PartEndEvent(inde

Traceback (most recent call last):
  File "/home/krishnendu/Research/fv-invariant-mining/.venv/lib64/python3.14/site-packages/IPython/core/interactiveshell.py", line 3745, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_33589/3529175581.py", line 11, in <module>
    async for event in aig_agent.run_stream_events(aig_str, deps=conf, usage_limits=limits):
    ...<2 lines>...
    		print(event)
  File "/home/krishnendu/Research/fv-invariant-mining/.venv/lib64/python3.14/site-packages/pydantic_ai/agent/abstract.py", line 986, in _run_stream_events
    result = await task
             ^^^^^^^^^^
  File "/home/krishnendu/Research/fv-invariant-mining/.venv/lib64/python3.14/site-packages/pydantic_ai/agent/abstract.py", line 961, in run_agent
    return await self.run(
           ^^^^^^^^^^^^^^^
    ...<15 lines>...
    )
    ^
  File "/home/krishnendu/Research/fv-invariant-mining/.venv/lib64/python3.14/site-packages/pydantic_ai/agent/abstract.py",

In [48]:
old_conf = conf

In [50]:
conf.score

-3

In [51]:
old_conf.strategies

['Check immediate parent-child relationships in AND gates - nodes that share common inputs often have equivalences',
 'Look for transitive equivalences: if A≡B and B≡C, then A≡C',
 'Check nodes that are structurally similar (same input patterns)']

In [52]:
conf.strategies

['Check immediate parent-child relationships in AND gates - nodes that share common inputs often have equivalences',
 'Look for transitive equivalences: if A≡B and B≡C, then A≡C',
 'Check nodes that are structurally similar (same input patterns)']

In [53]:
conf.knowledge_base

[]

In [44]:
print(conf.score)
print(conf.results)
print(conf.tried_tuples)
print(conf.strategies)
print(conf.knowledge_base)

0
set()
{(110, 108), (224, 222), (246, 244), (32, 14), (50, 16), (174, 152), (152, 136), (244, 196), (76, 52), (240, 238), (212, 192), (50, 30), (148, 144), (62, 24)}
['Check immediate parent-child relationships in AND gates - nodes that share common inputs often have equivalences', 'Look for transitive equivalences: if A≡B and B≡C, then A≡C', 'Check nodes that are structurally similar (same input patterns)']
[]


In [ ]:
nv, _ = conf.aig.simulate(128)

In [23]:
conf.tried_tuples

{(18, 16),
 (20, 12),
 (20, 16),
 (22, 10),
 (22, 18),
 (22, 20),
 (24, 22),
 (28, 26),
 (32, 30),
 (36, 34),
 (40, 38),
 (44, 42),
 (70, 2),
 (70, 6),
 (70, 14),
 (72, 24)}

In [21]:
conf.strategies


['Check immediate parent-child relationships in AND gates - nodes that share common inputs often have equivalences',
 'Look for transitive equivalences: if A≡B and B≡C, then A≡C',
 'Check nodes that are structurally similar (same input patterns)']

In [ ]:
old_conf = conf

In [ ]:
AIG.__dict__['handle_aig'].

<function aig_grapher.AIG.handle_aig(self, aiger_file: str | pathlib.Path) -> None>